In [7]:
import os
import sys
import json
import torch
import numpy as np

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), "../.."))
FEVER_DIR   = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.join(FEVER_DIR, "data")
RESULTS_DIR = os.path.join(FEVER_DIR, "results")

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Fever dir:   {FEVER_DIR}")
print(f"Data dir:    {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"GPU:         {torch.cuda.get_device_name(0)}")
print(f"VRAM free:   {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

Fever dir:   /home/jovyan/lectures/raq-reproducibility-challenge/fever
Data dir:    /home/jovyan/lectures/raq-reproducibility-challenge/fever/data
Results dir: /home/jovyan/lectures/raq-reproducibility-challenge/fever/results
GPU:         NVIDIA A40
VRAM free:   42.4 GB


In [8]:
from datasets import load_dataset

print("Loading FEVER dataset...")
dataset = load_dataset("copenlu/fever_gold_evidence")

def clean_fever_title(title):
    title = title.replace("-LRB-", "(")
    title = title.replace("-RRB-", ")")
    title = title.replace("-LSB-", "[")
    title = title.replace("-RSB-", "]")
    title = title.replace("-LCB-", "{")
    title = title.replace("-RCB-", "}")
    title = title.replace("_", " ")
    return title.strip()

# collect all unique article titles across all splits
all_articles = set()
for split in ["train", "validation", "test"]:
    for example in dataset[split]:
        for ev in example["evidence"]:
            if ev[0]:
                all_articles.add(clean_fever_title(ev[0]))

# build lookup set — lowercase for matching
titles_to_keep = {t.lower(): t for t in all_articles}

print(f"Unique articles to fetch: {len(titles_to_keep):,}")

Loading FEVER dataset...
Unique articles to fetch: 29,756


In [9]:
from datasets import load_dataset

PASSAGES_PATH = os.path.join(DATA_DIR, "fever_passages.jsonl")

# check if already done from previous attempt
if os.path.exists(PASSAGES_PATH):
    with open(PASSAGES_PATH) as f:
        existing = sum(1 for _ in f)
    print(f"passages file already exists with {existing:,} passages")
    print("Delete it and rerun this cell if you want to rebuild")
else:
    print("Streaming wikimedia/wikipedia — no full download needed...")
    print("This filters 29K articles from the full English Wikipedia")
    print("Progress printed every 100K articles checked\n")

    wiki_dataset = load_dataset(
        "wikimedia/wikipedia",
        "20231101.en",
        split="train",
        streaming=True
    )

    passages     = []
    found_titles = set()
    checked      = 0

    with open(PASSAGES_PATH, "w") as f:
        for article in wiki_dataset:
            checked += 1

            if checked % 100000 == 0:
                print(f"  Checked: {checked:,} | "
                      f"Found: {len(found_titles):,} / "
                      f"{len(titles_to_keep):,}")

            title_lower = article["title"].lower()
            if title_lower not in titles_to_keep:
                continue

            found_titles.add(title_lower)

            # split into 100-word chunks — same as the paper
            words  = article["text"].split()
            chunks = [
                words[i:i + 100]
                for i in range(0, len(words), 100)
            ]

            for chunk_id, chunk in enumerate(chunks):
                if len(chunk) < 10:
                    continue
                passage = {
                    "id":    f"{article['title']}_{chunk_id}",
                    "title": article["title"],
                    "text":  " ".join(chunk)
                }
                passages.append(passage)
                f.write(json.dumps(passage) + "\n")

            # stop early if we found everything
            if len(found_titles) >= len(titles_to_keep):
                print(f"\nFound all {len(found_titles):,} articles — stopping early")
                break

    print(f"\nDone!")
    print(f"  Checked:        {checked:,} Wikipedia articles")
    print(f"  Found:          {len(found_titles):,} / {len(titles_to_keep):,}")
    print(f"  Not found:      {len(titles_to_keep) - len(found_titles):,}")
    print(f"  Total passages: {len(passages):,}")
    print(f"  Saved to:       {PASSAGES_PATH}")

    # save not-found titles for reference
    not_found = [
        titles_to_keep[t] for t in titles_to_keep
        if t not in found_titles
    ]
    with open(os.path.join(DATA_DIR, "not_found_articles.json"), "w") as f:
        json.dump(not_found, f, indent=2)
    print(f"  Not found list: not_found_articles.json")

Streaming wikimedia/wikipedia — no full download needed...
This filters 29K articles from the full English Wikipedia
Progress printed every 100K articles checked



README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

  Checked: 100,000 | Found: 927 / 29,756
  Checked: 200,000 | Found: 1,492 / 29,756
  Checked: 300,000 | Found: 1,982 / 29,756
  Checked: 400,000 | Found: 2,386 / 29,756
  Checked: 500,000 | Found: 2,818 / 29,756
  Checked: 600,000 | Found: 3,256 / 29,756
  Checked: 700,000 | Found: 3,640 / 29,756
  Checked: 800,000 | Found: 4,029 / 29,756
  Checked: 900,000 | Found: 4,325 / 29,756
  Checked: 1,000,000 | Found: 4,645 / 29,756
  Checked: 1,100,000 | Found: 4,955 / 29,756
  Checked: 1,200,000 | Found: 5,185 / 29,756
  Checked: 1,300,000 | Found: 5,549 / 29,756
  Checked: 1,400,000 | Found: 5,842 / 29,756
  Checked: 1,500,000 | Found: 6,252 / 29,756
  Checked: 1,600,000 | Found: 6,591 / 29,756
  Checked: 1,700,000 | Found: 6,925 / 29,756
  Checked: 1,800,000 | Found: 7,201 / 29,756
  Checked: 1,900,000 | Found: 7,522 / 29,756
  Checked: 2,000,000 | Found: 7,812 / 29,756
  Checked: 2,100,000 | Found: 8,102 / 29,756
  Checked: 2,200,000 | Found: 8,420 / 29,756
  Checked: 2,300,000 | Found: 

In [23]:
# Reload passages from file
import json
import numpy as np

passages = []
with open(PASSAGES_PATH) as f:
    for line in f:
        passages.append(json.loads(line))

print(f"Total passages loaded: {len(passages):,}")
print("\nSample passages:")
for p in passages[:3]:
    print(f"\n  ID:    {p['id']}")
    print(f"  Title: {p['title']}")
    print(f"  Text:  {p['text'][:150]}...")

lengths = [len(p["text"].split()) for p in passages]
print(f"\nPassage length stats:")
print(f"  Mean:   {np.mean(lengths):.0f} words")
print(f"  Min:    {np.min(lengths)} words")
print(f"  Max:    {np.max(lengths)} words")

Total passages loaded: 574,197

Sample passages:

  ID:    Anarchism_0
  Title: Anarchism
  Text:  Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims...

  ID:    Anarchism_1
  Title: Anarchism
  Text:  organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anar...

  ID:    Anarchism_2
  Title: Anarchism
  Text:  anarchism. In the last decades of the 20th and into the 21st century, the anarchist movement has been resurgent once more, growing in popularity and i...

Passage length stats:
  Mean:   98 words
  Min:    10 words
  Max:    100 words


In [25]:
# Encode all 574K passages with DPR
EMBEDDINGS_PATH = os.path.join(DATA_DIR, "fever_embeddings.npy")

if os.path.exists(EMBEDDINGS_PATH):
    print(f"Embeddings already exist at {EMBEDDINGS_PATH}")
    print("Loading from disk...")
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Loaded embeddings shape: {embeddings.shape}")
else:
    BATCH_SIZE   = 512
    all_embeds   = []
    total        = len(passages)
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Encoding {total:,} passages")
    print(f"Batch size:     {BATCH_SIZE}")
    print(f"Total batches:  {total_batches:,}")
    print(f"Device:         {next(ctx_encoder.parameters()).device}")
    print(f"Estimated time: 20-30 min on A40\n")

    ctx_encoder.eval()
    with torch.no_grad():
        for i in range(0, total, BATCH_SIZE):
            batch = passages[i:i + BATCH_SIZE]

            # progress every 50 batches
            if (i // BATCH_SIZE) % 50 == 0:
                pct  = i / total * 100
                done = i // BATCH_SIZE
                print(f"  Batch {done:,}/{total_batches:,} "
                      f"({pct:.1f}%) | "
                      f"passages encoded: {i:,}")

            # tokenize
            encoded = ctx_tokenizer(
                [p["title"] for p in batch],
                [p["text"]  for p in batch],
                max_length=100,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            # encode on GPU
            outputs = ctx_encoder(
                input_ids=encoded["input_ids"].to("cuda"),
                attention_mask=encoded["attention_mask"].to("cuda")
            )

            # collect embeddings on CPU
            all_embeds.append(
                outputs.pooler_output.cpu().numpy()
            )

    # stack into single array
    embeddings = np.vstack(all_embeds)
    print(f"\nEncoding complete!")
    print(f"  Shape: {embeddings.shape}")
    print(f"  dtype: {embeddings.dtype}")
    print(f"  Size:  {embeddings.nbytes / 1024**3:.2f} GB")

    # save to disk
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"  Saved to: {EMBEDDINGS_PATH}")

Encoding 574,197 passages
Batch size:     512
Total batches:  1,122
Device:         cuda:0
Estimated time: 20-30 min on A40

  Batch 0/1,122 (0.0%) | passages encoded: 0
  Batch 50/1,122 (4.5%) | passages encoded: 25,600
  Batch 100/1,122 (8.9%) | passages encoded: 51,200
  Batch 150/1,122 (13.4%) | passages encoded: 76,800
  Batch 200/1,122 (17.8%) | passages encoded: 102,400
  Batch 250/1,122 (22.3%) | passages encoded: 128,000
  Batch 300/1,122 (26.8%) | passages encoded: 153,600
  Batch 350/1,122 (31.2%) | passages encoded: 179,200
  Batch 400/1,122 (35.7%) | passages encoded: 204,800
  Batch 450/1,122 (40.1%) | passages encoded: 230,400
  Batch 500/1,122 (44.6%) | passages encoded: 256,000
  Batch 550/1,122 (49.0%) | passages encoded: 281,600
  Batch 600/1,122 (53.5%) | passages encoded: 307,200
  Batch 650/1,122 (58.0%) | passages encoded: 332,800
  Batch 700/1,122 (62.4%) | passages encoded: 358,400
  Batch 750/1,122 (66.9%) | passages encoded: 384,000
  Batch 800/1,122 (71.3%) 

In [26]:
# Cell 7 - build FAISS index
import faiss
import transformers.utils.import_utils as import_utils
import transformers.utils as tu

# faiss patch
if hasattr(import_utils.is_faiss_available, "cache_clear"):
    import_utils.is_faiss_available.cache_clear()
import_utils._faiss_available = True
import_utils.is_faiss_available = lambda: True
tu.is_faiss_available = lambda: True

FAISS_INDEX_PATH = os.path.join(DATA_DIR, "fever_faiss.index")

if os.path.exists(FAISS_INDEX_PATH):
    print(f"FAISS index already exists at {FAISS_INDEX_PATH}")
    index = faiss.read_index(FAISS_INDEX_PATH)
    print(f"Loaded index with {index.ntotal:,} vectors")
else:
    print("Building FAISS index...")
    d = embeddings.shape[1]
    print(f"  Embedding dim: {d}")
    print(f"  Num passages:  {embeddings.shape[0]:,}")

    # normalise for cosine similarity via inner product
    embeds_norm = embeddings.copy().astype("float32")
    faiss.normalize_L2(embeds_norm)
    print(f"  Embeddings normalised")

    # build flat inner product index on GPU
    res       = faiss.StandardGpuResources()
    cpu_index = faiss.IndexFlatIP(d)
    gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

    gpu_index.add(embeds_norm)
    print(f"  Vectors added: {gpu_index.ntotal:,}")

    # move back to CPU and save
    index = faiss.index_gpu_to_cpu(gpu_index)
    faiss.write_index(index, FAISS_INDEX_PATH)

    print(f"\nFAISS index built and saved!")
    print(f"  Vectors:  {index.ntotal:,}")
    print(f"  Saved to: {FAISS_INDEX_PATH}")

    # check file size
    size_gb = os.path.getsize(FAISS_INDEX_PATH) / 1024**3
    print(f"  File size: {size_gb:.2f} GB")

Building FAISS index...
  Embedding dim: 768
  Num passages:  574,197
  Embeddings normalised
  Vectors added: 574,197

FAISS index built and saved!
  Vectors:  574,197
  Saved to: /home/jovyan/lectures/raq-reproducibility-challenge/fever/data/fever_faiss.index
  File size: 1.64 GB


In [27]:
# Cell 8 - sanity check search
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizerFast
)

print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizerFast.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to("cuda")
q_encoder.eval()
print("Question encoder loaded\n")

def search_index(claim, index, passages, q_encoder,
                 q_tokenizer, n_docs=5):
    """Search FAISS index for a given claim."""
    encoded = q_tokenizer(
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=300
    )
    with torch.no_grad():
        query_vec = q_encoder(
            input_ids=encoded["input_ids"].to("cuda"),
            attention_mask=encoded["attention_mask"].to("cuda")
        ).pooler_output.cpu().numpy()

    # normalise query vector — must match how we built the index
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(
        query_vec.astype("float32"), n_docs
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "rank":  len(results) + 1,
            "score": float(score),
            "title": passages[idx]["title"],
            "text":  passages[idx]["text"],
            "idx":   int(idx)
        })
    return results

# test with 3 FEVER claims we know the gold evidence for
test_claims = [
    {
        "claim":        "Barack Obama was born in Hawaii.",
        "label":        "SUPPORTS",
        "gold_article": "Barack Obama"
    },
    {
        "claim":        "The Eiffel Tower is located in Berlin.",
        "label":        "REFUTES",
        "gold_article": "Eiffel Tower"
    },
    {
        "claim":        "Cristiano Ronaldo is a professional footballer.",
        "label":        "SUPPORTS",
        "gold_article": "Cristiano Ronaldo"
    }
]

for tc in test_claims:
    print("=" * 65)
    print(f"Claim:        {tc['claim']}")
    print(f"Label:        {tc['label']}")
    print(f"Gold article: {tc['gold_article']}")
    print(f"\nTop 5 retrieved passages:")

    results = search_index(
        tc["claim"], index, passages,
        q_encoder, q_tokenizer, n_docs=5
    )

    gold_lower = tc["gold_article"].lower()
    for r in results:
        match = "✓" if gold_lower in r["title"].lower() else " "
        print(f"\n  [{match}] Rank {r['rank']} | "
              f"Score: {r['score']:.3f}")
        print(f"      Title: {r['title']}")
        print(f"      Text:  {r['text'][:120]}...")
    print()

Loading DPR question encoder...


Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Question encoder loaded

Claim:        Barack Obama was born in Hawaii.
Label:        SUPPORTS
Gold article: Barack Obama

Top 5 retrieved passages:

  [✓] Rank 1 | Score: 0.714
      Title: Barack Obama
      Text:  tier of American presidents. Early life and career Obama was born on August 4, 1961, at Kapiolani Medical Center for Wom...

  [✓] Rank 2 | Score: 0.702
      Title: Barack Obama
      Text:  participated in as a member of his high school's varsity team, and he is left-handed. In 2005, the Obama family applied ...

  [✓] Rank 3 | Score: 0.697
      Title: Barack Obama
      Text:  was killed in an automobile accident in 1982, when Obama was 21 years old. Recalling his early childhood, Obama said: "T...

  [✓] Rank 4 | Score: 0.696
      Title: Barack Obama
      Text:  descended from John Punch, an enslaved African man who lived in the Colony of Virginia during the seventeenth century. O...

  [ ] Rank 5 | Score: 0.686
      Title: John McCain
      Text:  and he was grant

In [28]:
# Cell 9 - save index metadata
import json

metadata = {
    "n_passages":       len(passages),
    "n_articles":       23733,
    "n_articles_total": 29756,
    "coverage_pct":     79.8,
    "embedding_dim":    768,
    "index_type":       "IndexFlatIP",
    "normalised":       True,
    "encoder":          "facebook/dpr-ctx_encoder-single-nq-base",
    "q_encoder":        "facebook/dpr-question_encoder-single-nq-base",
    "chunk_size_words": 100,
    "wikipedia_dump":   "wikimedia/wikipedia 20231101.en",
    "paper_dump":       "Wikipedia December 2018",
    "passages_path":    PASSAGES_PATH,
    "embeddings_path":  EMBEDDINGS_PATH,
    "faiss_index_path": FAISS_INDEX_PATH,
    "sanity_check": {
        "obama_top1_correct":   True,
        "ronaldo_top1_correct": True,
        "eiffel_top1_correct":  False,
        "eiffel_note": (
            "REFUTES claim — retriever fetches Berlin passages "
            "because claim mentions Berlin prominently. "
            "Known DPR behaviour on false claims."
        )
    }
}

META_PATH = os.path.join(DATA_DIR, "index_metadata.json")
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved")
print(json.dumps(metadata, indent=2))

Metadata saved
{
  "n_passages": 574197,
  "n_articles": 23733,
  "n_articles_total": 29756,
  "coverage_pct": 79.8,
  "embedding_dim": 768,
  "index_type": "IndexFlatIP",
  "normalised": true,
  "encoder": "facebook/dpr-ctx_encoder-single-nq-base",
  "q_encoder": "facebook/dpr-question_encoder-single-nq-base",
  "chunk_size_words": 100,
  "wikipedia_dump": "wikimedia/wikipedia 20231101.en",
  "paper_dump": "Wikipedia December 2018",
  "passages_path": "/home/jovyan/lectures/raq-reproducibility-challenge/fever/data/fever_passages.jsonl",
  "embeddings_path": "/home/jovyan/lectures/raq-reproducibility-challenge/fever/data/fever_embeddings.npy",
  "faiss_index_path": "/home/jovyan/lectures/raq-reproducibility-challenge/fever/data/fever_faiss.index",
  "sanity_check": {
    "obama_top1_correct": true,
    "ronaldo_top1_correct": true,
    "eiffel_top1_correct": false,
    "eiffel_note": "REFUTES claim \u2014 retriever fetches Berlin passages because claim mentions Berlin prominently. Know